# EDA y Preprocesamiento — Dataset Amazon
## PA02 — Analítica de Datos | Grupo 05 | USS 2026-I

In [ ]:
# ============================================================
# CELDA 2: Importación de librerías
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Librerías importadas correctamente")

In [ ]:
# ============================================================
# CELDA 3: Carga del dataset
# ============================================================
df = pd.read_csv('data/Amazon.csv')

print("✅ Dataset cargado correctamente")
print(f"  Filas    : {df.shape[0]:,}")
print(f"  Columnas : {df.shape[1]}")

In [ ]:
# ============================================================
# CELDA 4: Exploración inicial
# ============================================================

# Tipos de datos
print("=" * 50)
print("TIPOS DE DATOS POR COLUMNA")
print("=" * 50)
print(df.dtypes)

# Valores nulos
print("\n" + "=" * 50)
print("VALORES NULOS POR COLUMNA")
print("=" * 50)
print(df.isnull().sum())

# Primeras filas
print("\n" + "=" * 50)
print("PRIMERAS 5 FILAS")
print("=" * 50)
df.head()

In [ ]:
# ============================================================
# CELDA 5: Estadísticas descriptivas
# ============================================================
columnas_numericas = ['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']

estadisticas = df[columnas_numericas].describe().round(2)

print("ESTADÍSTICAS DESCRIPTIVAS — VARIABLES NUMÉRICAS")
print(estadisticas)

In [ ]:
# ============================================================
# CELDA 6: Gráfica 1 — Distribución de OrderStatus
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))

conteo = df['OrderStatus'].value_counts()
colores = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']

bars = ax.bar(conteo.index, conteo.values, color=colores,
              edgecolor='white', linewidth=1.5)

# Etiquetas encima de cada barra
for bar, valor in zip(bars, conteo.values):
    porcentaje = valor / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{valor:,}\n({porcentaje:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Distribución de Estados de Pedidos — Amazon (2020–2024)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Estado del Pedido', fontsize=12)
ax.set_ylabel('Número de Pedidos', fontsize=12)
ax.set_ylim(0, max(conteo.values) * 1.2)

plt.tight_layout()
plt.savefig('graficas/grafica1_distribucion_estados.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica 1 guardada en /graficas")

In [ ]:
# ============================================================
# CELDA 7: Gráfica 2 — Cancelaciones por método de pago
# ============================================================

# Crear columna binaria temporal para visualización
df['EsCancelado'] = (df['OrderStatus'] == 'Cancelled').astype(int)

cancelaciones_pago = df.groupby('PaymentMethod')['EsCancelado'].agg(
    Total='count',
    Cancelados='sum'
).reset_index()

cancelaciones_pago['TasaCancelacion'] = (
    cancelaciones_pago['Cancelados'] / cancelaciones_pago['Total'] * 100
).round(2)

cancelaciones_pago = cancelaciones_pago.sort_values('TasaCancelacion', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cancelaciones_pago['PaymentMethod'],
               cancelaciones_pago['TasaCancelacion'],
               color='#e74c3c', edgecolor='white', linewidth=1.5)

for bar, valor in zip(bars, cancelaciones_pago['TasaCancelacion']):
    ax.text(bar.get_width() + 0.05,
            bar.get_y() + bar.get_height()/2,
            f'{valor}%', va='center', fontsize=11, fontweight='bold')

ax.set_title('Tasa de Cancelación por Método de Pago',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tasa de Cancelación (%)', fontsize=12)
ax.set_ylabel('Método de Pago', fontsize=12)
ax.set_xlim(0, max(cancelaciones_pago['TasaCancelacion']) * 1.3)

plt.tight_layout()
plt.savefig('graficas/grafica2_cancelaciones_pago.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica 2 guardada en /graficas")

In [ ]:
# ============================================================
# CELDA 8: Gráfica 3 — Mapa de calor de correlaciones
# ============================================================
fig, ax = plt.subplots(figsize=(8, 6))

correlaciones = df[columnas_numericas].corr().round(2)
mask = np.triu(np.ones_like(correlaciones, dtype=bool))

sns.heatmap(correlaciones, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True,
            linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})

ax.set_title('Mapa de Calor — Correlaciones entre Variables Numéricas',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('graficas/grafica3_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica 3 guardada en /graficas")

In [ ]:
# ============================================================
# CELDA 9: Gráfica 4 — Distribución de variables numéricas clave
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# TotalAmount
axes[0].hist(df['TotalAmount'], bins=50, color='#3498db',
             edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribución de TotalAmount', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Monto Total (USD)', fontsize=11)
axes[0].set_ylabel('Frecuencia', fontsize=11)
axes[0].axvline(df['TotalAmount'].mean(), color='red', linestyle='--',
                linewidth=2, label=f"Media: ${df['TotalAmount'].mean():.2f}")
axes[0].legend()

# ShippingCost
axes[1].hist(df['ShippingCost'], bins=50, color='#2ecc71',
             edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribución de ShippingCost', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Costo de Envío (USD)', fontsize=11)
axes[1].set_ylabel('Frecuencia', fontsize=11)
axes[1].axvline(df['ShippingCost'].mean(), color='red', linestyle='--',
                linewidth=2, label=f"Media: ${df['ShippingCost'].mean():.2f}")
axes[1].legend()

plt.suptitle('Distribución de Variables Numéricas Clave',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('graficas/grafica4_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfica 4 guardada en /graficas")

In [ ]:
# ============================================================
# CELDA 10: Crear variable target binaria
# ============================================================
# 1 = Cancelado, 0 = No cancelado
df['Target'] = (df['OrderStatus'] == 'Cancelled').astype(int)

print("DISTRIBUCIÓN DE LA VARIABLE TARGET")
print("=" * 40)
print(df['Target'].value_counts())
print(f"\nPorcentaje cancelados   : {df['Target'].mean()*100:.2f}%")
print(f"Porcentaje no cancelados: {(1-df['Target'].mean())*100:.2f}%")

In [ ]:
# ============================================================
# CELDA 11: One-Hot Encoding de variables categóricas
# ============================================================

# Columnas que se descartan (identificadores, no informativas)
columnas_descartar = ['OrderID', 'OrderDate', 'CustomerID', 'CustomerName',
                      'ProductID', 'ProductName', 'SellerID',
                      'OrderStatus', 'EsCancelado']

# Columnas que se codifican
columnas_categoricas = ['Category', 'PaymentMethod', 'Brand', 'Country']

# Eliminar columnas no útiles
df_modelo = df.drop(columns=columnas_descartar)

# Aplicar One-Hot Encoding
df_modelo = pd.get_dummies(df_modelo, columns=columnas_categoricas, drop_first=False)

print(f"Columnas antes del encoding : 20")
print(f"Columnas después del encoding: {df_modelo.shape[1]}")
print(f"Filas: {df_modelo.shape[0]:,}")
print("\nPrimeras columnas del dataset procesado:")
print(list(df_modelo.columns[:15]))

In [ ]:
# ============================================================
# CELDA 12: Estandarización de variables numéricas
# ============================================================

# Separar features y target
X = df_modelo.drop(columns=['Target'])
y = df_modelo['Target']

# Columnas numéricas a escalar
cols_escalar = ['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']

# Aplicar StandardScaler
scaler = StandardScaler()
X[cols_escalar] = scaler.fit_transform(X[cols_escalar])

print("✅ Estandarización aplicada")
print(f"\nMedia de UnitPrice después del escalado : {X['UnitPrice'].mean():.4f} (debe ser ~0)")
print(f"Std  de UnitPrice después del escalado : {X['UnitPrice'].std():.4f}  (debe ser ~1)")

In [ ]:
# ============================================================
# CELDA 13: División train/test y guardado del dataset
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # mantiene la proporción de cancelados en ambos conjuntos
)

print("DIVISIÓN DEL DATASET")
print("=" * 40)
print(f"Entrenamiento : {X_train.shape[0]:,} filas ({len(X_train)/len(X)*100:.0f}%)")
print(f"Prueba        : {X_test.shape[0]:,} filas ({len(X_test)/len(X)*100:.0f}%)")
print(f"\nCancelados en entrenamiento: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Cancelados en prueba       : {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

# Guardar para usar en Fase 3
import joblib
joblib.dump(X_train, 'modelos/X_train.pkl')
joblib.dump(X_test, 'modelos/X_test.pkl')
joblib.dump(y_train, 'modelos/y_train.pkl')
joblib.dump(y_test, 'modelos/y_test.pkl')
joblib.dump(scaler, 'modelos/scaler.pkl')

print("\n✅ Datos guardados en /modelos — listos para Fase 3")